<a href="https://cognitiveclass.ai"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0101EN-SkillsNetwork/images/IDSN-logo.png" width="400"> </a>

<h1 align=center><font size = 5>Regression Models with Keras</font></h1>


## Introduction


Como comentamos en los videos, a pesar de la popularidad de bibliotecas más potentes como PyToch y TensorFlow, no son fáciles de usar y tienen una curva de aprendizaje pronunciada. Por lo tanto, para las personas que recién comienzan a aprender sobre aprendizaje profundo, no hay mejor biblioteca que Keras.

Keras es una API de alto nivel para crear modelos de aprendizaje profundo. Ha ganado popularidad por su facilidad de uso y simplicidad sintáctica, lo que facilita un desarrollo rápido. Como verá en este laboratorio y en los otros laboratorios de este curso, se puede crear una red de aprendizaje profundo muy compleja con Keras con solo unas pocas líneas de código. Apreciará Keras aún más una vez que aprenda a crear modelos profundos con PyTorch y TensorFlow en los otros cursos.

Por lo tanto, en este laboratorio, aprenderá a usar la biblioteca Keras para crear un modelo de regresión.


<h2>Modelos de regresión con Keras</h2>

<h3>Objetivo de este cuaderno<h3>
<h5>1. Cómo usar la biblioteca Keras para crear un modelo de regresión.</h5>
<h5>2. Descargar y limpiar el conjunto de datos.</h5>
<h5>3. Crear una red neuronal.</h5>
<h5>4. Entrenar y probar la red.</h5>


## Table of Contents

<div class="alert alert-block alert-info" style="margin-top: 20px">

<font size = 3>
    
1. <a href="#item31">Download and Clean Dataset</a>  
2. <a href="#item32">Import Keras</a>  
3. <a href="#item33">Build a Neural Network</a>  
4. <a href="#item34">Train and Test the Network</a>  

</font>
</div>


<a id="item31"></a>


## Download and Clean Dataset


Let's start by importing the <em>pandas</em> and the Numpy libraries.


In [ ]:
# All Libraries required for this lab are listed below. The libraries pre-installed on Skills Network Labs are commented. 
# If you run this notebook on a different environment, e.g. your desktop, you may need to uncomment and install certain libraries.

#!pip install numpy==1.21.4
#!pip install pandas==1.3.4
#!pip install keras==2.1.6

In [ ]:
import pandas as pd
import numpy as np

import warnings
warnings.simplefilter('ignore', FutureWarning)

Jugaremos con el mismo conjunto de datos que usamos en los videos.

<strong>El conjunto de datos trata sobre la resistencia a la compresión de diferentes muestras de hormigón en función de los volúmenes de los diferentes ingredientes que se usaron para fabricarlas. Los ingredientes incluyen:</strong>

<strong>1. Cement</strong>

<strong>2. Blast Furnace Slag</strong>

<strong>3. Fly Ash</strong>

<strong>4. Water</strong>

<strong>5. Superplasticizer</strong>

<strong>6. Coarse Aggregate</strong>

<strong>7. Fine Aggregate</strong>


Let's download the data and read it into a <em>pandas</em> dataframe.


In [ ]:
concrete_data = pd.read_csv('https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0101EN/labs/data/concrete_data.csv')
concrete_data.head()

La primera muestra de hormigón contiene 540 metros cúbicos de cemento, 0 metros cúbicos de escoria de alto horno, 0 metros cúbicos de cenizas volantes, 162 metros cúbicos de agua, 2,5 metros cúbicos de superplastificante, 1040 metros cúbicos de árido grueso y 676 metros cúbicos de árido fino. Esta mezcla de hormigón, que tiene 28 días de antigüedad, tiene una resistencia a la compresión de 79,99 MPa.


#### Let's check how many data points we have.


In [ ]:
concrete_data.shape

Por lo tanto, hay aproximadamente 1000 muestras para entrenar nuestro modelo. Debido a que son pocas, debemos tener cuidado de no sobreajustar los datos de entrenamiento.

Verifiquemos el conjunto de datos para ver si faltan valores.


In [ ]:
concrete_data.describe()

In [ ]:
concrete_data.isnull().sum()

Los datos se ven muy limpios y están listos para ser utilizados para construir nuestro modelo.

#### Split data into predictors and target
#### Dividir los datos en predictores y objetivos

La variable objetivo en este problema es la resistencia de la muestra de hormigón. Por lo tanto, nuestros predictores serán todas las demás columnas.


In [ ]:
concrete_data_columns = concrete_data.columns

predictors = concrete_data[concrete_data_columns[concrete_data_columns != 'Strength']] # all columns except Strength
target = concrete_data['Strength'] # Strength column

<a id="item2"></a>


Hagamos una rápida comprobación de la cordura de los predictores y los marcos de datos de destino.


In [ ]:
predictors.head()

In [ ]:
target.head()

Finalmente, el último paso es normalizar los datos restando la media y dividiéndolos por la desviación estándar.


In [ ]:
predictors_norm = (predictors - predictors.mean()) / predictors.std()
predictors_norm.head()

Guardemos el número de predictores en *n_cols* ya que necesitaremos este número al construir nuestra red.


In [ ]:
n_cols = predictors_norm.shape[1] # number of predictors

<a id="item1"></a>


<a id='item32'></a>


## Import Keras


Recuerda que en los videos Keras normalmente se ejecuta sobre una biblioteca de bajo nivel como TensorFlow. Esto significa que para poder usar la biblioteca Keras, primero tendrás que instalar TensorFlow y cuando importes la biblioteca Keras, se mostrará explícitamente qué backend se usó para instalar la biblioteca Keras. En CC Labs, usamos TensorFlow como backend para instalar Keras, por lo que debería mostrarse claramente cuando importemos Keras.


#### Let's go ahead and import the Keras library


In [ ]:
import keras

Como puede ver, se utilizó el backend de TensorFlow para instalar la biblioteca Keras.


Importemos el resto de los paquetes de la biblioteca Keras que necesitaremos para construir nuestro modelo de regresión.

In [ ]:
from keras.models import Sequential
from keras.layers import Dense

<a id='item33'></a>


## Build a Neural Network


Definamos una función que defina nuestro modelo de regresión para que podamos llamarla cómodamente para crear nuestro modelo.


In [ ]:
# define regression model
def regression_model():
    # create model
    model = Sequential()
    model.add(Dense(50, activation='relu', input_shape=(n_cols,)))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(1))
    
    # compile model
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

The above function create a model that has two hidden layers, each of 50 hidden units.


<a id="item4"></a>


<a id='item34'></a>


## Train and Test the Network


Let's call the function now to create our model.


In [ ]:
# build the model
model = regression_model()

Next, we will train and test the model at the same time using the *fit* method. We will leave out 30% of the data for validation and we will train the model for 100 epochs.


In [ ]:
# fit the model
model.fit(predictors_norm, target, validation_split=0.3, epochs=100, verbose=2)

<strong>You can refer to this [link](https://keras.io/models/sequential/) to learn about other functions that you can use for prediction or evaluation.</strong>


Feel free to vary the following and note what impact each change has on the model's performance:

1. Increase or decreate number of neurons in hidden layers
2. Add more hidden layers
3. Increase number of epochs


### Thank you for completing this lab!

This notebook was created by [Alex Aklson](https://www.linkedin.com/in/aklson/). I hope you found this lab interesting and educational. Feel free to contact me if you have any questions!



## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2020-09-21  | 2.0  | Srishti  |  Migrated Lab to Markdown and added to course repo in GitLab |



<hr>

## <h3 align="center"> © IBM Corporation 2020. All rights reserved. <h3/>


This notebook is part of a course on **Coursera** called *Introduction to Deep Learning & Neural Networks with Keras*. If you accessed this notebook outside the course, you can take this course online by clicking [here](https://cocl.us/DL0101EN_Coursera_Week3_LAB1).


<hr>

Copyright &copy; 2019 [IBM Developer Skills Network](https://cognitiveclass.ai/?utm_source=bducopyrightlink&utm_medium=dswb&utm_campaign=bdu). This notebook and its source code are released under the terms of the [MIT License](https://bigdatauniversity.com/mit-license/).
